In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# LTAU 하이퍼파라미터(ltau_bins, ltau_k) 그리드 서치 결과를 비교한다.
# 참고: run_cue_ltau_param_1.sh / run_cue_ltau_param_2.sh
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# run_cue_ltau_param_1.sh / _2.sh의 LTAU_BINS_LIST / LTAU_K_LIST와 동일하게 맞춘다.
LTAU_BINS_LIST = [1250, 2500, 5000, 10000]
LTAU_K_LIST = [3, 5, 7]

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(dataset, target, bins, k):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc를 뽑아온다."""
    model_dir = os.path.join(cue_base, dataset, target, 'v1', f'LTAU_BIN{bins}_K{k}')
    metrics_path = os.path.join(model_dir, 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for bins in LTAU_BINS_LIST:
            for k in LTAU_K_LIST:
                vals = load_metric(d_name, target, bins, k)
                if vals is not None:
                    raw_rows.append({'Dataset': d_name, 'Target': target, 'BIN': bins, 'K': k, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다 (기대값: 2 x 4 x 12 = 96).")

✅ 데이터 로드 완료: 96개의 항목이 수집되었습니다 (기대값: 2 x 4 x 12 = 96).


In [3]:
def plot_ltau_heatmap(df_all, dataset, target_name, metric_name, output_dir):
    """LTAU_BINS(x축) x LTAU_K(y축) 조합에 대한 metric 값을 heatmap으로 표시한다."""
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target_name)]
    if df_sub.empty or metric_name not in df_sub.columns:
        return

    # 4(bins) x 3(k) 행렬 구성
    grid = pd.DataFrame(index=LTAU_K_LIST, columns=LTAU_BINS_LIST, dtype=float)
    for bins in LTAU_BINS_LIST:
        for k in LTAU_K_LIST:
            row = df_sub[(df_sub['BIN'] == bins) & (df_sub['K'] == k)]
            if not row.empty:
                grid.loc[k, bins] = row[metric_name].values[0]

    if grid.isna().all(axis=None):
        return

    # rmse는 낮을수록 좋으므로 반전 colormap을, r2/pcc는 높을수록 좋으므로 정방향 colormap을 사용한다.
    cmap = 'RdYlGn_r' if LOWER_IS_BETTER.get(metric_name, True) else 'RdYlGn'

    fig, ax = plt.subplots(figsize=(6, 4.5))
    sns.heatmap(grid, annot=True, fmt='.3f', cmap=cmap, ax=ax, cbar_kws={'label': metric_name.upper()})

    ax.set_xlabel('LTAU_BINS')
    ax.set_ylabel('LTAU_K')
    ax.set_title(f"{dataset} / {target_name} / {metric_name}")

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'ltau_param_comparison')

for d_name in datasets:
    for target in targets:
        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            plot_ltau_heatmap(df_total, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/DIS/PI_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/DIS/PI_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/DIS/PI_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/DIS/Ssel_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/DIS/Ssel_pcc.png


... (14 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/INH/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/INH/RS(k=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/INH/RS(k=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/ltau_param_comparison/INH/RS(k=2)_pcc.png
